# Pokemon TCG AI Battle — Submission Monitor

This notebook wraps the `kaggle` CLI's simulation-competition commands (see
[kaggle-cli/docs/simulation_competitions.md](https://github.com/Kaggle/kaggle-cli/blob/main/docs/simulation_competitions.md))
so you can submit your agent and monitor it without memorizing CLI flags.

**Competition:** [pokemon-tcg-ai-battle](https://www.kaggle.com/competitions/pokemon-tcg-ai-battle)

**Before you start:**
1. `pip install kaggle` (or make sure it's already installed and up to date).
2. Put your Kaggle API token at `~/.kaggle/kaggle.json` (Kaggle account settings → *Create New Token*).
3. Go to the [competition page](https://www.kaggle.com/competitions/pokemon-tcg-ai-battle) and click
   **"I Understand and Accept"** — the API will reject downloads/submissions until you've joined.

Every command below is run through one small `run()` helper so you can see exactly what CLI call is
being made, and every "list" style command is parsed into a `pandas.DataFrame` where possible so it's
easy to sort/filter/plot.


In [1]:
import subprocess, shlex, time, json, io
from datetime import datetime
import pandas as pd

pd.set_option("display.max_colwidth", 80)

COMPETITION = "pokemon-tcg-ai-battle"

def run(args, check=True):
    '''Run a `kaggle ...` CLI command, print it, and return (stdout, stderr).

    `args` is a list of CLI arguments *after* `kaggle`, e.g. ["competitions", "submissions", COMPETITION].
    '''
    cmd = ["kaggle"] + args
    print("$", " ".join(shlex.quote(a) for a in cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {' '.join(cmd)}")
    return result.stdout, result.stderr


def run_to_df(args, csv_flag="-v"):
    '''Run a `kaggle` command with the CSV flag and parse stdout into a DataFrame.

    Falls back to a best-effort whitespace parse if the command doesn't support -v/--csv
    (not every subcommand does — if this fails, just call `run()` directly and read the
    printed table instead).
    '''
    cmd_args = list(args)
    if csv_flag and csv_flag not in cmd_args:
        cmd_args = cmd_args + [csv_flag]
    stdout, stderr = run(cmd_args, check=False)
    if not stdout.strip():
        return pd.DataFrame()
    try:
        return pd.read_csv(io.StringIO(stdout))
    except Exception:
        print("[info] Could not parse output as CSV — returning raw text instead.")
        return stdout


## 0. Sanity check: CLI installed, credentials found, competition joined

`kaggle competitions list --group entered` should include `pokemon-tcg-ai-battle`. If it doesn't,
go accept the rules on the competition page first — everything below will fail with a 403 otherwise.


In [3]:
run(["--version"])
run(["competitions", "list", "--group", "entered"])


$ kaggle --version
Kaggle CLI 2.2.3

$ kaggle competitions list --group entered
ref                                                        deadline             category      reward  teamCount  userHasEntered  
---------------------------------------------------------  -------------------  --------  ----------  ---------  --------------  
https://www.kaggle.com/competitions/pokemon-tcg-ai-battle  2026-08-16 23:59:00  Featured   Knowledge       4514            True  
https://www.kaggle.com/competitions/orbit-wars             2026-07-07 23:59:00  Featured  50,000 Usd       4729            True  



('ref                                                        deadline             category      reward  teamCount  userHasEntered  \n---------------------------------------------------------  -------------------  --------  ----------  ---------  --------------  \nhttps://www.kaggle.com/competitions/pokemon-tcg-ai-battle  2026-08-16 23:59:00  Featured   Knowledge       4514            True  \nhttps://www.kaggle.com/competitions/orbit-wars             2026-07-07 23:59:00  Featured  50,000 Usd       4729            True  \n',
 '')

## 1. Inspect the competition

Read the rules/evaluation pages and skim the discussion forum for tips before you iterate.


In [4]:
# List available competition pages (description, rules, evaluation, data-description, ...)
run(["competitions", "pages", COMPETITION])


$ kaggle competitions pages pokemon-tcg-ai-battle
name                               
---------------------------------  
data-description                   
Description                        
Evaluation                         
rules                              
abstract                           
Timeline                           
Prizes                             
How to Play Pokémon TCG            
How to Submit to this Competition  
Trademark Note                     
Frequently Asked Questions         



('name                               \n---------------------------------  \ndata-description                   \nDescription                        \nEvaluation                         \nrules                              \nabstract                           \nTimeline                           \nPrizes                             \nHow to Play Pokémon TCG            \nHow to Submit to this Competition  \nTrademark Note                     \nFrequently Asked Questions         \n',
 '')

In [5]:
# Read the full text of a specific page, e.g. the evaluation criteria
run(["competitions", "pages", COMPETITION, "--content"])


$ kaggle competitions pages pokemon-tcg-ai-battle --content
name                               content                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

('name                               content                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [ ]:
# Top discussion topics — good place to check for environment quirks / common bugs
topics_df = run_to_df(["competitions", "topics", "list", COMPETITION, "-s", "top", "--page-size", "10"])
topics_df


In [ ]:
# Read a specific topic (and its comments) once you've picked an id from the table above
TOPIC_ID = None  # e.g. 123456

if TOPIC_ID is not None:
    run(["competitions", "topics", "show", COMPETITION, str(TOPIC_ID)])
else:
    print("Set TOPIC_ID above to read a specific discussion thread.")


## 2. Download competition data (environment, starter kit, sample agents)

In [6]:
run(["competitions", "download", COMPETITION, "-p", f"{COMPETITION}-data"])


$ kaggle competitions download pokemon-tcg-ai-battle -p pokemon-tcg-ai-battle-data



  0%|          | 0.00/301M [00:00<?, ?B/s]
  0%|          | 1.00M/301M [00:00<02:43, 1.92MB/s]
  1%|          | 2.00M/301M [00:00<02:20, 2.23MB/s]
  1%|          | 3.00M/301M [00:01<02:12, 2.35MB/s]
  1%|▏         | 4.00M/301M [00:01<02:07, 2.44MB/s]
  2%|▏         | 5.00M/301M [00:02<02:05, 2.48MB/s]
  2%|▏         | 6.00M/301M [00:02<02:02, 2.52MB/s]
  2%|▏         | 7.00M/301M [00:03<02:02, 2.52MB/s]
  3%|▎         | 8.00M/301M [00:03<02:02, 2.52MB/s]
  3%|▎         | 9.00M/301M [00:03<02:00, 2.54MB/s]
  3%|▎         | 10.0M/301M [00:04<02:00, 2.53MB/s]
  4%|▎         | 11.0M/301M [00:04<01:58, 2.56MB/s]
  4%|▍         | 12.0M/301M [00:05<01:59, 2.55MB/s]
  4%|▍         | 13.0M/301M [00:05<01:58, 2.54MB/s]
  5%|▍         | 14.0M/301M [00:05<01:57, 2.56MB/s]
  5%|▍         | 15.0M/301M [00:06<01:57, 2.55MB/s]
  5%|▌         | 16.0M/301M [00:06<01:57, 2.54MB/s]
  6%|▌         | 17.0M/301M [00:07<01:5

('Downloading pokemon-tcg-ai-battle.zip to pokemon-tcg-ai-battle-data\n\n',
 '\n  0%|          | 0.00/301M [00:00<?, ?B/s]\n  0%|          | 1.00M/301M [00:00<02:43, 1.92MB/s]\n  1%|          | 2.00M/301M [00:00<02:20, 2.23MB/s]\n  1%|          | 3.00M/301M [00:01<02:12, 2.35MB/s]\n  1%|▏         | 4.00M/301M [00:01<02:07, 2.44MB/s]\n  2%|▏         | 5.00M/301M [00:02<02:05, 2.48MB/s]\n  2%|▏         | 6.00M/301M [00:02<02:02, 2.52MB/s]\n  2%|▏         | 7.00M/301M [00:03<02:02, 2.52MB/s]\n  3%|▎         | 8.00M/301M [00:03<02:02, 2.52MB/s]\n  3%|▎         | 9.00M/301M [00:03<02:00, 2.54MB/s]\n  3%|▎         | 10.0M/301M [00:04<02:00, 2.53MB/s]\n  4%|▎         | 11.0M/301M [00:04<01:58, 2.56MB/s]\n  4%|▍         | 12.0M/301M [00:05<01:59, 2.55MB/s]\n  4%|▍         | 13.0M/301M [00:05<01:58, 2.54MB/s]\n  5%|▍         | 14.0M/301M [00:05<01:57, 2.56MB/s]\n  5%|▍         | 15.0M/301M [00:06<01:57, 2.55MB/s]\n  5%|▌         | 16.0M/301M [00:06<01:57, 2.54MB/s]\n  6%|▌         | 17.0M/301M 

## 3. Submit your agent

- **Single file agent**: just point `-f` at your `main.py`.
- **Multi-file agent**: bundle everything into a `submission.tar.gz` with `main.py` at the root first.

Give every submission a distinct `-m` message (e.g. a version tag) so you can tell them apart later.


In [ ]:
def submit_agent(filepath, message):
    '''Submit a single-file or .tar.gz agent to the competition.'''
    return run(["competitions", "submit", COMPETITION, "-f", filepath, "-m", message])

# Single-file example:
# submit_agent("main.py", "v1")

# Multi-file example — bundle first, then submit the archive:
# subprocess.run(["tar", "-czf", "submission.tar.gz", "main.py", "helper.py", "model_weights.pkl"], check=True)
# submit_agent("submission.tar.gz", "v1 multi-file")


## 4. Check submission status

Lists all of your submissions to this competition, most recent first. The `id` (or `ref`, depending
on the CLI version) column is what you need for the episode/replay/log commands below — grab it from
the printed table or the parsed DataFrame.


In [7]:
submissions_df = run_to_df(["competitions", "submissions", COMPETITION])
submissions_df


$ kaggle competitions submissions pokemon-tcg-ai-battle -v
ref,fileName,date,description,status,publicScore,privateScore

54437175,v1.tar.gz,2026-07-07 18:15:32.440000,,SubmissionStatus.COMPLETE,474.3,




,ref,fileName,date,description,status,publicScore,privateScore
0,54437175,v1.tar.gz,2026-07-07 18:15:32.440000,NaN,SubmissionStatus.COMPLETE,474.3,NaN


## 5. List episodes for a submission

Once a submission has played some games, list its episodes. Replace `SUBMISSION_ID` with a value
from the `submissions_df` table above.


In [8]:
SUBMISSION_ID = 54437175  # e.g. 12345678

def list_episodes(submission_id):
    return run_to_df(["competitions", "episodes", str(submission_id)])

if SUBMISSION_ID is not None:
    episodes_df = list_episodes(SUBMISSION_ID)
    episodes_df
else:
    print("Set SUBMISSION_ID above (from the submissions table) to list its episodes.")


$ kaggle competitions episodes 54437175 -v
id,createTime,endTime,state,type

84683954,2026-07-07 18:19:49.205000,2026-07-07 18:22:27.300000,EpisodeState.COMPLETED,EpisodeType.EPISODE_TYPE_PUBLIC

84683096,2026-07-07 18:15:32.948000,2026-07-07 18:18:22.133000,EpisodeState.COMPLETED,EpisodeType.EPISODE_TYPE_VALIDATION


Use "kaggle competitions replay <episode_id>" to download a replay, or "kaggle competitions logs <episode_id> <agent_index>" for agent logs.



## 6. Poll loop — watch a submission until it has results

Simulation-competition submissions take a while to accumulate episodes and a public score. This loop
re-checks on an interval and stops once new episodes show up (or you hit `max_checks`). Interrupt the
kernel any time to stop early — nothing here is destructive.


In [ ]:
def monitor_submission(submission_id, interval_seconds=300, max_checks=12):
    '''Poll `kaggle competitions episodes <submission_id>` every `interval_seconds`
    until the episode count changes or `max_checks` is reached.
    '''
    seen_episode_count = -1
    for i in range(max_checks):
        print(f"--- check {i + 1}/{max_checks} @ {datetime.now().isoformat(timespec='seconds')} ---")
        df = list_episodes(submission_id)
        n = len(df) if isinstance(df, pd.DataFrame) else 0
        print(f"episodes so far: {n}")
        if n != seen_episode_count and seen_episode_count != -1:
            print("New episode activity detected.")
        seen_episode_count = n
        if i < max_checks - 1:
            time.sleep(interval_seconds)
    return list_episodes(submission_id)

# Example (uncomment to run — this will block the kernel while it polls):
final_episodes_df = monitor_submission(SUBMISSION_ID, interval_seconds=300, max_checks=12)


--- check 1/12 @ 2026-07-07T20:28:29 ---
$ kaggle competitions episodes 54437175 -v
id,createTime,endTime,state,type

84683954,2026-07-07 18:19:49.205000,2026-07-07 18:22:27.300000,EpisodeState.COMPLETED,EpisodeType.EPISODE_TYPE_PUBLIC

84683096,2026-07-07 18:15:32.948000,2026-07-07 18:18:22.133000,EpisodeState.COMPLETED,EpisodeType.EPISODE_TYPE_VALIDATION


Use "kaggle competitions replay <episode_id>" to download a replay, or "kaggle competitions logs <episode_id> <agent_index>" for agent logs.

episodes so far: 3
--- check 2/12 @ 2026-07-07T20:33:30 ---
$ kaggle competitions episodes 54437175 -v
id,createTime,endTime,state,type

84684739,2026-07-07 18:23:42.109000,2026-07-07 18:31:26.284000,EpisodeState.COMPLETED,EpisodeType.EPISODE_TYPE_PUBLIC

84685377,2026-07-07 18:28:11.985000,2026-07-07 18:30:35.799000,EpisodeState.COMPLETED,EpisodeType.EPISODE_TYPE_PUBLIC

84683954,2026-07-07 18:19:49.205000,2026-07-07 18:22:27.300000,EpisodeState.COMPLETED,EpisodeType.EPISODE_TYPE_PUBLIC

846

## 7. Download a replay / agent logs for a specific episode

Grab an `episode_id` from the episodes table above. Agent index is 0-based (0 = first agent in the
match, 1 = second, etc).


In [ ]:
EPISODE_ID = 84683954   # e.g. 98765432
AGENT_INDEX = 0

if EPISODE_ID is not None:
    run(["competitions", "replay", str(EPISODE_ID), "-p", "./replays"])
    run(["competitions", "logs", str(EPISODE_ID), str(AGENT_INDEX), "-p", "./logs"])
else:
    print("Set EPISODE_ID above to download its replay and logs.")


## 8. Check the leaderboard

In [ ]:
leaderboard_df = run_to_df(["competitions", "leaderboard", COMPETITION, "-s"])
leaderboard_df


## 9. Scout the leader

Grab a `teamId` from the leaderboard table above, list that team's active agents, pick the one with
the best public score, and inspect its episodes/replays/logs the same way you would your own.


In [ ]:
LEADER_TEAM_ID = None  # e.g. 42

if LEADER_TEAM_ID is not None:
    team_df = run_to_df(["competitions", "team-submissions", str(LEADER_TEAM_ID)])
    display(team_df)

    if isinstance(team_df, pd.DataFrame) and not team_df.empty and "publicScore" in team_df.columns:
        best_submission_id = team_df.sort_values("publicScore", ascending=False).iloc[0]["id"]
        print(f"Best submission for team {LEADER_TEAM_ID}: {best_submission_id}")
        best_episodes_df = list_episodes(int(best_submission_id))
        display(best_episodes_df)
else:
    print("Set LEADER_TEAM_ID above (from the leaderboard table) to scout them.")


## Appendix: full loop as a single cell

Everything above, chained together, matching the "Putting It All Together" workflow from the
kaggle-cli docs. Edit the variables at the top and run.


In [ ]:
# --- Config ---------------------------------------------------------------
AGENT_FILE = "main.py"
SUBMIT_MESSAGE = "v1"
DO_DOWNLOAD_DATA = True
DO_SUBMIT = False          # flip to True once you're ready
DO_MONITOR = False         # flip to True to poll after submitting

# --- 1. Data + discussion ---------------------------------------------------
if DO_DOWNLOAD_DATA:
    run(["competitions", "download", COMPETITION, "-p", f"{COMPETITION}-data"])

topics_df = run_to_df(["competitions", "topics", "list", COMPETITION, "-s", "top", "--page-size", "10"])
display(topics_df)

# --- 2. Submit ---------------------------------------------------------------
if DO_SUBMIT:
    submit_agent(AGENT_FILE, SUBMIT_MESSAGE)

# --- 3. Check status -----------------------------------------------------
submissions_df = run_to_df(["competitions", "submissions", COMPETITION])
display(submissions_df)

# --- 4. Monitor the newest submission ---------------------------------------
if DO_MONITOR and isinstance(submissions_df, pd.DataFrame) and not submissions_df.empty:
    id_col = "id" if "id" in submissions_df.columns else submissions_df.columns[0]
    newest_id = submissions_df.iloc[0][id_col]
    monitor_submission(int(newest_id), interval_seconds=300, max_checks=6)

# --- 5. Leaderboard + scout the leader ---------------------------------------
leaderboard_df = run_to_df(["competitions", "leaderboard", COMPETITION, "-s"])
display(leaderboard_df)
